In [ ]:
!nvidia-smi

Sat Oct 10 10:41:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P0             28W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install music21 pretty_midi pyfluidsynth
!apt-get install -y fluidsynth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.7 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-common at-spi2-core fluid-soundfont-gm gsettings-desktop-schemas
  libatk-bridge2.0-0t64 libatk1.0-0t64 libatspi2.0-0t64 libb2-1
  libdouble-conversion3 libevdev2 libfluidsynth3 libgtk-3-0t64 libgtk-3-bin
  libgtk-3-common libgudev-1.0-0 libinput-bin libinput10 libinstpatch-1.0-2
  libmd4c0 libmtdev1t64 libpipewire-0.3-0t64 libpipewire-0.3-common
  libproxy1v5 libqt6core6t64 libqt6dbus6t64 libqt6gui6t64 libqt6network6t64
  libqt6opengl6t64 libqt6qml6 libqt6qmlmodels6 libqt6quick6
  libqt6waylandclient6 libqt6waylandcompositor6
  libqt6waylandeglclienthwintegration6
  libqt6waylandeglcompositorhwintegration6 libqt6widgets6t64
  libqt6wlshellintegration6 librsvg2-common libspa-0

In [ ]:
import tensorflow as tf
import music21
import pretty_midi

print("TensorFlow:", tf.__version__)
print("music21:", music21.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
music21: 9.9.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!mkdir -p /content/midi_data
!unzip -q "/content/drive/MyDrive/music_data/archive (1).zip" -d /content/midi_data

In [4]:
import glob

midi_files = glob.glob("/content/midi_data/**/*.mid", recursive=True) + \
             glob.glob("/content/midi_data/**/*.midi", recursive=True) + \
             glob.glob("/content/midi_data/**/*.MID", recursive=True)

print("Total MIDI files:", len(midi_files))
print("Examples:")
for f in midi_files[:5]:
    print(f)

Total MIDI files: 295
Examples:
/content/midi_data/balakir/islamei.mid
/content/midi_data/mendelssohn/mendel_op53_5.mid
/content/midi_data/mendelssohn/mendel_op30_5.mid
/content/midi_data/mendelssohn/mendel_op30_2.mid
/content/midi_data/mendelssohn/mendel_op30_3.mid


In [5]:
from music21 import converter, note, chord

# Pehli file load karo
midi = converter.parse(midi_files[0])
print("File:", midi_files[0])

# Is file ke saare notes aur chords nikalo
elements = midi.flatten().notes

# Pehle 20 dikhao
for el in list(elements)[:20]:
    if isinstance(el, note.Note):
        print("Note :", el.pitch)
    elif isinstance(el, chord.Chord):
        print("Chord:", [str(p) for p in el.pitches])

/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, data=b'Copyright \xa9 2000 by Bernd Kr\xfcger'>; getting generic Instrument
  warnings.warn(


File: /content/midi_data/balakir/islamei.mid
Note : B-3
Note : B-3
Note : B-3
Note : B-3
Note : A3
Note : B-3
Note : F#3
Note : F#3
Note : A3
Note : B-3
Note : C#4
Note : C4
Note : G#3
Note : G#3
Note : G#3
Note : G#3
Note : F3
Note : F#3
Note : G#3
Note : B-3


In [6]:
import random, pickle
from music21 import converter, note, chord

random.seed(42)
selected_files = random.sample(midi_files, min(150, len(midi_files)))

all_notes = []
failed = 0

for i, file in enumerate(selected_files):
    try:
        midi = converter.parse(file)
        for el in midi.flatten().notes:
            if isinstance(el, note.Note):
                all_notes.append(str(el.pitch))
            elif isinstance(el, chord.Chord):
                all_notes.append('.'.join(str(n) for n in el.normalOrder))
    except Exception as e:
        failed += 1
    if (i + 1) % 10 == 0:
        print(f"Processed {i+1}/{len(selected_files)} files")

print("\nTotal notes/chords collected:", len(all_notes))
print("Failed files:", failed)
print("Unique notes/chords:", len(set(all_notes)))

# Drive mein save karo
with open('/content/drive/MyDrive/music_data/notes.pkl', 'wb') as f:
    pickle.dump(all_notes, f)
print("Saved to Drive: notes.pkl")

/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, data=b'Copyright \xa9 1997 by Bernd Krueger'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyright \xa9 2010 by Bernd Krueger'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, data=b'Produced 1997 by Bernd Kr\xfcger.'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, data=b'Copyrig

Processed 10/150 files


/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyright \xa9 2009 by Bernd Krueger'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=6, data=b'Copyright \xa9 2010 by Bernd Krueger'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=b'Von fremden L\xe4ndern und Menschen'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyr

Processed 20/150 files


/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=6, data=b'Copyright \xa9 2000 by Bernd Kr\xfcger'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, data=b'Copyright \xa9 2002 by Bernd Kr\xfcger'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyright \xa9 2001 by Bernd Kr\xfcger'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=

...

Processed 130/150 files


/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=b'Schumann:   Fr\xf6hlicher Landmann, von der Arbeit zur\xfcckkehrend'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=b'aus Album f\xfcr die Jugend Opus 68, Nr. 10'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, data=b'Schumann: Fr\xf6hlicher Landmann'>; getting generic Instrument
  warnings.warn(


Processed 140/150 files


/usr/local/lib/python3.13/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=b'Grieg: Lyrische St\xfccke Book II Opus 38 Nr. 4 - Halling'>; getting generic Instrument
  warnings.warn(


Processed 150/150 files

Total notes/chords collected: 239806
Failed files: 0
Unique notes/chords: 529
Saved to Drive: notes.pkl


In [7]:
import numpy as np
import pickle

SEQ_LENGTH = 50

# 1. Vocabulary banao
pitch_names = sorted(set(all_notes))
n_vocab = len(pitch_names)
note_to_int = {n: i for i, n in enumerate(pitch_names)}
int_to_note = {i: n for n, i in note_to_int.items()}

# 2. Sequences banao
network_input = []
network_output = []

for i in range(len(all_notes) - SEQ_LENGTH):
    seq_in = all_notes[i : i + SEQ_LENGTH]
    seq_out = all_notes[i + SEQ_LENGTH]
    network_input.append([note_to_int[n] for n in seq_in])
    network_output.append(note_to_int[seq_out])

X = np.array(network_input)
y = np.array(network_output)

print("Vocabulary size (n_vocab):", n_vocab)
print("Total training sequences:", len(X))
print("X shape:", X.shape)
print("y shape:", y.shape)

# 3. Vocabulary Drive mein save karo
with open('/content/drive/MyDrive/music_data/vocab.pkl', 'wb') as f:
    pickle.dump({'note_to_int': note_to_int,
                 'int_to_note': int_to_note,
                 'n_vocab': n_vocab,
                 'seq_length': SEQ_LENGTH}, f)
print("Saved to Drive: vocab.pkl")

Vocabulary size (n_vocab): 529
Total training sequences: 239756
X shape: (239756, 50)
y shape: (239756,)
Saved to Drive: vocab.pkl


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=n_vocab, output_dim=100),
    LSTM(256, return_sequences=True),
    Dropout(0.3),
    LSTM(256),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(n_vocab, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

# Model ko ek baar input shape de do taake summary poori dikhe
model.build(input_shape=(None, SEQ_LENGTH))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 100)        │        52,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 50, 256)        │       365,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 529)            │       135,953 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,145,525 (4.37 MB)

 Trainable params: 1,145,525 (4.37 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/music_data/music_model.keras',
    monitor='loss',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(monitor='loss', patience=4, restore_best_weights=True)

callbacks = [checkpoint, early_stop]
print("Callbacks ready")

Callbacks ready


In [10]:
history = model.fit(
    X, y,
    epochs=20,
    batch_size=256,
    callbacks=callbacks
)

Epoch 1/20
937/937 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 4.8389
Epoch 1: loss improved from None to 4.65462, saving model to /content/drive/MyDrive/music_data/music_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/music_data/music_model.keras
937/937 ━━━━━━━━━━━━━━━━━━━━ 33s 30ms/step - loss: 4.6546
Epoch 2/20
937/937 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 4.3830
Epoch 2: loss improved from 4.65462 to 4.29263, saving model to /content/drive/MyDrive/music_data/music_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/music_data/music_model.keras
937/937 ━━━━━━━━━━━━━━━━━━━━ 38s 30ms/step - loss: 4.2926
Epoch 3/20
936/937 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 4.0048
Epoch 3: loss improved from 4.29263 to 3.94633, saving model to /content/drive/MyDrive/music_data/music_model.keras

Epoch 3: finished saving model to /content/drive/MyDrive/music_data/music_model.keras
937/937 ━━━━━━━━━━━━━━━━━━━━ 29s 31ms/step - loss: 3.9463
Epoch 4/20
937/9

In [1]:
import numpy as np

def sample_with_temperature(probs, temperature=1.0):
    probs = np.asarray(probs).astype('float64')
    probs = np.log(probs + 1e-9) / temperature
    probs = np.exp(probs)
    probs = probs / np.sum(probs)
    return np.random.choice(len(probs), p=probs)

def generate_notes(num_notes=300, temperature=0.8):
    start = np.random.randint(0, len(X))
    pattern = list(X[start])
    output = []
    for _ in range(num_notes):
        inp = np.array(pattern).reshape(1, SEQ_LENGTH)
        preds = model.predict(inp, verbose=0)[0]
        idx = sample_with_temperature(preds, temperature)
        output.append(int_to_note[idx])
        pattern.append(idx)
        pattern = pattern[1:]
    return output

print("Functions ready")

Functions ready


In [2]:
generated = generate_notes(num_notes=300, temperature=0.8)

print("Total generated:", len(generated))
print("First 20:", generated[:20])

NameError: name 'X' is not defined

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pickle
import numpy as np
from tensorflow.keras.models import load_model

# Notes aur vocabulary load karo
with open('/content/drive/MyDrive/music_data/notes.pkl', 'rb') as f:
    all_notes = pickle.load(f)

with open('/content/drive/MyDrive/music_data/vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

note_to_int = vocab['note_to_int']
int_to_note = vocab['int_to_note']
n_vocab = vocab['n_vocab']
SEQ_LENGTH = vocab['seq_length']

# X dobara banao
network_input = []
for i in range(len(all_notes) - SEQ_LENGTH):
    seq_in = all_notes[i : i + SEQ_LENGTH]
    network_input.append([note_to_int[n] for n in seq_in])
X = np.array(network_input)

# Trained model load karo
model = load_model('/content/drive/MyDrive/music_data/music_model.keras')

print("n_vocab:", n_vocab)
print("X shape:", X.shape)
print("Model loaded")

Mounted at /content/drive
n_vocab: 529
X shape: (239756, 50)
Model loaded


In [4]:
generated = generate_notes(num_notes=300, temperature=0.8)

print("Total generated:", len(generated))
print("First 20:", generated[:20])

Total generated: 300
First 20: ['E-5', 'C5', 'E-5', 'C#4', 'B5', 'C#5', 'G#5', 'E5', 'G#5', 'G#4', 'D6', 'G#5', 'E6', 'E4', 'B6', 'D4', 'A6', 'B3', 'F#6', 'A4']


In [5]:
!pip install -q music21 pretty_midi pyfluidsynth
!apt-get install -y -q fluidsynth
print("Libraries ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.4 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-common at-spi2-core fluid-soundfont-gm gsettings-desktop-schemas
  libatk-bridge2.0-0t64 libatk1.0-0t64 libatspi2.0-0t64 libb2-1
  libdouble-conversion3 libevdev2 libfluidsynth3 libgtk-3-0t64 libgtk-3-bin
  libgtk-3-common libgudev-1.0-0 libinput-bin libinput10 libinstpatch-1.0-2
  libmd4c0 libmtdev1t64 libpipewire-0.3-0t64 libpipewire-0.3-common
  libproxy1v5 libqt6core6t64 libqt6dbus6t64 libqt6gui6t64 libqt6network6t64
  libqt6opengl6t64 libqt6qml6 libqt6qmlmodels6 libqt6quick6
  libqt6waylandclient6 libqt6waylandcompositor6
  libqt6waylandeglclienthwintegration6
  libqt6waylandeglcompositorhwintegration6 libqt6widgets6t64
  libqt6wlshellintegration6 librsvg2-common libspa-0.2-modules libt

In [6]:
from music21 import stream, note, chord, instrument

def notes_to_midi(note_list, filename):
    offset = 0
    output_notes = []

    for pattern in note_list:
        # Chord (jaise "4.7" ya "0.4.7" ya sirf "5")
        if ('.' in pattern) or pattern.isdigit():
            chord_notes = []
            for n in pattern.split('.'):
                new_note = note.Note(int(n))
                new_note.storedInstrument = instrument.Piano()
                chord_notes.append(new_note)
            new_chord = chord.Chord(chord_notes)
            new_chord.offset = offset
            output_notes.append(new_chord)
        # Single note (jaise "E4")
        else:
            new_note = note.Note(pattern)
            new_note.offset = offset
            new_note.storedInstrument = instrument.Piano()
            output_notes.append(new_note)

        offset += 0.5

    midi_stream = stream.Stream(output_notes)
    midi_stream.write('midi', fp=filename)
    print("MIDI saved:", filename)

midi_path = '/content/drive/MyDrive/music_data/generated_music.mid'
notes_to_midi(generated, midi_path)

MIDI saved: /content/drive/MyDrive/music_data/generated_music.mid


In [7]:
import pretty_midi
from IPython.display import Audio, display
from scipy.io import wavfile

pm = pretty_midi.PrettyMIDI(midi_path)
audio = pm.fluidsynth(fs=44100)

wav_path = '/content/drive/MyDrive/music_data/generated_music.wav'
wavfile.write(wav_path, 44100, (audio * 32767).astype('int16'))
print("WAV saved:", wav_path)

display(Audio(audio, rate=44100))

WAV saved: /content/drive/MyDrive/music_data/generated_music.wav
